## Topic 4: Different Types of Chunking & Text Splitters

### Concept, Intuition, Why It Exists

- Topic 3 covered chunking *strategies* (fixed-size, sentence-aware, semantic, structure-aware) as conceptual approaches. This topic covers **text splitters** — the actual reusable, named implementations of those strategies that show up repeatedly across RAG tooling (LangChain and similar libraries), worth knowing by name because interviewers and real codebases reference them directly, not just the underlying concept.
- A "splitter" is just a specific, parameterized implementation of a chunking strategy, usually built to be swappable — same input Document in, different splitter, different chunk boundaries out, with no other part of the pipeline needing to change. This is the same Document-abstraction payoff as the loaders: splitters are interchangeable precisely because they all consume and produce the same shape.
- **CharacterTextSplitter**: splits on a single fixed separator (commonly a blank line or newline) once content exceeds chunk size, falling back to a hard character cut if that separator never appears within the limit. It's the library-level cousin of fixed-size chunking from Topic 3, with one specific improvement — it tries a separator before giving up and cutting blindly.
- **RecursiveCharacterTextSplitter**: tries a *list* of separators in priority order, falling back from the most meaning-preserving separator to the least. Covered in full depth in the next topic, since it's the most commonly used default in practice and deserves its own dedicated treatment.
- **TokenTextSplitter**: splits based on actual token count (using the same tokenizer the target LLM/embedding model uses) rather than character count. This directly addresses the "token count vs. character count" hidden concept flagged back in Topic 1 — a chunk that looks fine by character count can still silently overflow a token-based context limit, and this splitter is built specifically to prevent that.
- **MarkdownHeaderTextSplitter** (and equivalents for HTML, code, etc.): a structure-aware splitter, but generalized beyond just Q&A pairs — it splits on a document's own formatting markers (`#`, `##`, `###` headers in Markdown; HTML tags; code function/class boundaries) the same way Topic 3's Q&A splitter exploited "Q:"/"A:" markers, just for a different kind of structure.

### Internal Working

- **CharacterTextSplitter**: scan the text for the configured separator; accumulate content between separator occurrences until the next addition would exceed `chunk_size`; if no separator occurrence exists within the size budget at all, cut at the raw character limit as a last resort.
- **TokenTextSplitter**: tokenize the full text first using the target tokenizer, then group tokens into chunks of the configured token count, decoding each group back into text — this guarantees every chunk respects the token budget exactly, unlike character-based splitters which only approximate it.
- **MarkdownHeaderTextSplitter**: parse the document for header markers, treat each header's section as a structural unit, and attach the header hierarchy itself into each chunk's metadata (e.g. "this chunk falls under Section 2 > Subsection 2.1") — giving retrieval extra structural context beyond just the chunk text itself, similar in spirit to how the JSON loader attaches `document_code`/`version`/`page` metadata.

### How It's Implemented In This Project

- This project's hand-written `chunk_text()`/`chunker.py` (Topic 2) is functionally closest to a sentence-aware variant of CharacterTextSplitter, built by hand rather than via a library, with overlap added explicitly.
- TokenTextSplitter-style logic becomes directly relevant the moment chunk sizing needs to be tied precisely to the embedding model's or LLM's actual token limit rather than an approximate character count — worth adopting if/when this project starts hitting silent token-limit truncation in production.
- MarkdownHeaderTextSplitter-style logic is directly applicable to any future Markdown-formatted SOPs or internal docs ingested into this project, the same way the Q&A structure-aware splitter applies to FAQ content — both are instances of "exploit existing document structure instead of inferring boundaries."

### Real-World Issues, Edge Cases, Debugging

- **CharacterTextSplitter's fallback is a silent quality cliff**: when the configured separator never appears within the chunk size budget (e.g. one extremely long paragraph with no blank lines), it falls back to a hard character cut with zero warning — producing exactly the syntactically-broken chunks this whole topic exists to avoid, but only for the specific documents unlucky enough to trigger the fallback. Worth explicitly checking for and flagging, not assuming the splitter always behaves the better way.
- **TokenTextSplitter requires picking the *right* tokenizer**: using a different model's tokenizer than the one actually being used downstream produces a chunk that's correctly sized for the wrong model — e.g. sizing chunks using one model's tokenizer while embedding with a completely different model's tokenizer means the resulting token counts can be meaningfully off for the model that actually matters.
- **MarkdownHeaderTextSplitter assumes well-formed, consistent headers**: a document with inconsistent header levels (skipping from `#` straight to `###`, or using bold text instead of real header markers in places) produces an inconsistent or incomplete structural hierarchy — the same brittleness failure mode flagged for Q&A structure-aware chunking in Topic 3, just applied to a different structural cue.
- **Mixing splitter types within one pipeline**: nothing prevents using different splitters for different source types in the same project (Markdown docs with the header splitter, FAQ content with the Q&A splitter, generic prose with the recursive splitter) — this is the recommended approach, not a complication to avoid, mirroring the same per-source-type decision already made at the loader level.

### Design Decisions & Trade-offs

- Library-provided splitters vs. hand-written chunking logic: this project currently hand-writes its chunker, trading a dependency for full visibility and control over exact behavior — reasonable while the team wants to understand every step deeply (as this whole learning sequence demonstrates), but a real production system more often reaches for well-tested library splitters to avoid re-solving already-solved edge cases (tokenizer alignment, separator fallback behavior) from scratch.
- Token-based vs. character-based sizing: token-based is strictly more correct for respecting an LLM's actual context limit, but requires running the tokenizer at chunk time, which is a real (if usually small) additional compute cost compared to just counting characters.

### Alternatives & When To Use Each

- CharacterTextSplitter — simple prose with a reliable separator (consistent paragraph breaks) and a tolerance for occasional fallback-cut chunks on unusually long sections.
- TokenTextSplitter — chunk size needs to map precisely to a model's actual context/embedding limit, not an approximation; worth the small extra tokenization cost when token-limit overflows are a measured real problem.
- MarkdownHeaderTextSplitter (or HTML/code equivalents) — source content already has reliable, consistent structural markers (headers, tags, function boundaries) to exploit directly, the same reasoning as Q&A structure-aware chunking in Topic 3.
- Hand-written custom logic (this project's current approach) — full control and visibility needed, team is building deep understanding of the mechanics rather than treating chunking as a black box.

### Common Mistakes & Production Failures

- Assuming character-based chunk sizes translate predictably to token counts, especially for non-English content or code, where the character-to-token ratio shifts significantly.
- Using a tokenizer mismatched to the actual downstream model when token-based splitting is adopted, producing chunks sized correctly for the wrong model.
- Applying a structure-aware splitter (Markdown headers, Q&A) to content that doesn't actually have consistent structure, and not validating extraction completeness afterward.
- Treating "which splitter to use" as a one-time, project-wide decision instead of a per-source-type decision, the same mistake flagged at the strategy level in Topic 3.

### Lead-Level Interview Questions

**Q: Why would you ever need a token-based splitter instead of just being more conservative with character-based chunk sizes?**
A: Because the character-to-token ratio isn't fixed — it varies by language, content type, and even specific vocabulary. Being "conservative" with character counts to be safe means either wasting context budget on content that tokenizes efficiently, or still risking overflow on content that tokenizes poorly (dense technical text, non-English content, code) — a token-based splitter removes the guesswork entirely by measuring the thing that actually matters.

**Q: A document's CharacterTextSplitter-based chunking silently falls back to hard character cuts on a few documents. How would you detect this happening in production without manually reading every chunk?**
A: Track, per document, whether the configured separator was actually found within the size budget or whether the fallback path was taken — log a flag or metric whenever the fallback fires, then monitor that count the same way other ingestion stages monitor warning counts. A spike signals specific documents (or document types) where the separator assumption doesn't hold, worth investigating rather than letting silently degrade.

**Q: When would you choose a Markdown/structure-aware splitter over a generic recursive splitter, even though the recursive splitter is more broadly applicable?**
A: When the source content has reliable, consistent structural markers that directly encode the ideal chunk boundaries — splitting on real headers preserves the document's own logical organization (and lets that hierarchy be attached as metadata for richer retrieval context) far more precisely than inferring boundaries purely from separators or size, the same reasoning that favored Q&A structure-aware chunking over generic chunking in Topic 3.

### Hidden Concepts Worth Knowing

- **Tokenizer alignment as a cross-cutting concern**: the same token-vs-character mismatch that affects chunk sizing here also affects cost estimation, context-window budgeting at query time, and prompt construction generally — it's not a chunking-specific issue, just one of the places it shows up first.
- **Metadata-enriched chunks**: structure-aware splitters that attach header hierarchy or section context into chunk metadata enable retrieval-time filtering and citation generation beyond what plain text-similarity search alone provides — e.g. "only search within Section 3" or "show the user which section this answer came from," both of which depend on that structural metadata being captured at chunk time, not reconstructed later.

### Revision Summary

> Beyond conceptual chunking strategies, several named, reusable splitter implementations exist: CharacterTextSplitter (single separator with hard-cut fallback), TokenTextSplitter (sizes by actual tokenizer output, not character approximation), and MarkdownHeaderTextSplitter (generalizes structure-aware chunking to any document with formatting markers, attaching structural metadata along the way). RecursiveCharacterTextSplitter — the most commonly used default — gets its own dedicated topic next.

In [1]:
"""
text_splitters.py
--------------------
Hand-written implementations of three named splitter types, built to be
directly comparable to chunker.py's sentence-aware splitter and the four
strategies in chunking_strategies.py.

  character_text_split  -- single separator, hard-cut fallback.
  token_text_split        -- sizes chunks by actual token count, not chars.
  markdown_header_split   -- splits on Markdown headers, attaching the
                              header hierarchy into each chunk's metadata.
"""

import re
from document import make_document


# ----------------------------------------------------------------------
# 1. CharacterTextSplitter -- single separator, hard-cut fallback
# ----------------------------------------------------------------------
def character_text_split(text: str, separator: str = "\n\n", chunk_size: int = 200) -> list:
    pieces = text.split(separator)
    chunks, current = [], ""

    for piece in pieces:
        candidate = (current + separator + piece) if current else piece
        if len(candidate) > chunk_size:
            if current:
                chunks.append(current)
            if len(piece) > chunk_size:
                # fallback: separator didn't help, hard-cut this piece
                for i in range(0, len(piece), chunk_size):
                    chunks.append(piece[i:i + chunk_size])
                current = ""
            else:
                current = piece
        else:
            current = candidate

    if current:
        chunks.append(current)
    return chunks


# ----------------------------------------------------------------------
# 2. TokenTextSplitter -- sizes by token count, not character count.
#    Uses a simple whitespace tokenizer as a stand-in; swap in a real
#    tokenizer (e.g. tiktoken, or the embedding model's own tokenizer)
#    to match this exactly to your actual downstream model.
# ----------------------------------------------------------------------
def _simple_tokenize(text: str) -> list:
    return text.split()  # placeholder tokenizer -- replace with a real one in production


def _simple_detokenize(tokens: list) -> str:
    return " ".join(tokens)


def token_text_split(text: str, chunk_size_tokens: int = 30) -> list:
    tokens = _simple_tokenize(text)
    return [
        _simple_detokenize(tokens[i:i + chunk_size_tokens])
        for i in range(0, len(tokens), chunk_size_tokens)
    ]


# ----------------------------------------------------------------------
# 3. MarkdownHeaderTextSplitter -- splits on # / ## / ### headers,
#    attaching the header hierarchy into each chunk's metadata.
# ----------------------------------------------------------------------
HEADER_RE = re.compile(r"^(#{1,3})\s+(.*)$", re.MULTILINE)


def markdown_header_split(text: str) -> list:
    matches = list(HEADER_RE.finditer(text))
    if not matches:
        return [make_document(text.strip(), {"header_path": []})]

    documents = []
    header_stack = []  # tracks current (level, title) hierarchy

    for i, match in enumerate(matches):
        level = len(match.group(1))
        title = match.group(2).strip()

        # pop any headers at the same or deeper level before pushing this one
        header_stack = [h for h in header_stack if h[0] < level]
        header_stack.append((level, title))

        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        section_text = text[start:end].strip()

        if section_text:
            documents.append(make_document(
                page_content=section_text,
                metadata={"header_path": [h[1] for h in header_stack]},
            ))

    return documents


if __name__ == "__main__":
    sample = (
        "Premature withdrawal incurs a 1 percent penalty.\n\n"
        "This does not apply if the FD is closed due to the death of the depositor.\n\n"
        "Senior citizens receive an additional 0.5 percent interest on all tenures."
    )

    print("--- CharacterTextSplitter ---")
    for c in character_text_split(sample, separator="\n\n", chunk_size=80):
        print(f"  {c!r}")

    print("\n--- TokenTextSplitter ---")
    for c in token_text_split(sample, chunk_size_tokens=10):
        print(f"  {c!r}")

    markdown_sample = (
        "# FD Policy\n"
        "Overview text about FD policy.\n"
        "## Premature Withdrawal\n"
        "A 1 percent penalty applies.\n"
        "### Death of Depositor Exception\n"
        "No penalty applies in this case.\n"
        "## Senior Citizen Rate\n"
        "An additional 0.5 percent applies.\n"
    )
    print("\n--- MarkdownHeaderTextSplitter ---")
    for d in markdown_header_split(markdown_sample):
        print(f"  header_path={d['metadata']['header_path']}: {d['page_content']!r}")

--- CharacterTextSplitter ---
  'Premature withdrawal incurs a 1 percent penalty.'
  'This does not apply if the FD is closed due to the death of the depositor.'
  'Senior citizens receive an additional 0.5 percent interest on all tenures.'

--- TokenTextSplitter ---
  'Premature withdrawal incurs a 1 percent penalty. This does not'
  'apply if the FD is closed due to the death'
  'of the depositor. Senior citizens receive an additional 0.5 percent'
  'interest on all tenures.'

--- MarkdownHeaderTextSplitter ---
  header_path=['FD Policy']: 'Overview text about FD policy.'
  header_path=['FD Policy', 'Premature Withdrawal']: 'A 1 percent penalty applies.'
  header_path=['FD Policy', 'Premature Withdrawal', 'Death of Depositor Exception']: 'No penalty applies in this case.'
  header_path=['FD Policy', 'Senior Citizen Rate']: 'An additional 0.5 percent applies.'
